# LVMark: four inpainting methods

Repository: [https://github.com/kuai-lab/TIFS26_lvmark](https://github.com/kuai-lab/TIFS26_lvmark)

Paper: [https://arxiv.org/abs/2412.09122](https://arxiv.org/abs/2412.09122)

Inspected commit: `da7ad67ba2d86ae9918dce2b42d3736e335561be`

Watermark stage: `in_generation`
Domain: video

This method is coupled to generation, so the notebook uses only author-provided or author-generated native data. It does not fake a MOSE embedding.

ProPainter, DiffuEraser, EffectErase, and E2FGVI-HQ receive the same authentic watermarked frames and binary mask. One official DiffuEraser execution is used: its internal ProPainter prior is preserved as the ProPainter result, and its final refinement is the DiffuEraser result. Inputs shorter than 22 frames are padded by repeating only the last frame/mask to 24 for that pipeline, then both reported outputs are trimmed to the authentic length. EffectErase applies its released Wan 4n+1 temporal adapter; E2FGVI-HQ uses the authors' official arbitrary-resolution test entry point.

**Release/resource blocker recorded on 2026-11-10:** the repository's inference shell script contains placeholders and refers to `prompts/512_ours` plus a prompt/key JSON that are not checked in. A documented one-sample replacement can be constructed, but the official 2.05 GB weight archive was served to Zeppelin at roughly 25 KB/s (more than 24 hours estimated), so the native watermark could not be generated during this run. The partial download was preserved for resume. Once the weights are available, the notebook runs the unchanged DynamiCrafter entry point, attacks its native output, and applies the released LVMark decoder/BCH record.


## Authenticity contract

- Watermark preparation: Run the official watermarked DynamiCrafter inference with the released LVMark weights, then provide a binary mask aligned to the generated video.
- Checkpoint status: REQUIRES the released LVMark weights and DynamiCrafter base checkpoint. The checked-in inference script also references a prompt-image directory and prompt/key JSON that are not present in the repository; the official weight archive index contains weight ZIPs, not those prompt files, so they must be obtained separately from the authors.
- Protocol note: LVMark jointly modifies a latent video diffusion decoder; it is not a post-hoc encoder for existing MOSE clips.
- No result cell fabricates metrics. Native extraction must finish and write its JSON before values are displayed.
- A method without a released spatial detector reports bit recovery only; it is not assigned a proxy localizer.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys
from IPython.display import JSON, Video, display

def find_repo_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "scripts/remaining_watermark_methods.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the paper-reproductions-task1 checkout")

ROOT = find_repo_root()
SCRIPTS = ROOT / "scripts"

def run(command, *, cwd=None, env=None):
    command = [str(value) for value in command]
    print("+", shlex.join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def require(path, description):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")
    return path

def require_commit(repository, expected):
    actual = subprocess.check_output(["git", "-C", str(repository), "rev-parse", "HEAD"], text=True).strip()
    if actual != expected:
        raise RuntimeError(f"Expected {expected} in {repository}, found {actual}")

def show_results(run_root):
    run_root = Path(run_root)
    for name in ("watermarked.mp4", "aligned-mask.mp4"):
        path = run_root / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    attack = run_root / "attack"
    for name in (
        "input-watermarked.mp4", "gt-mask.mp4", "propainter.mp4", "diffueraser.mp4",
        "effecterase.mp4", "e2fgvi.mp4",
    ):
        path = attack / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    result_json = {
        *run_root.rglob("summary.json"),
        *(attack / name for name in ("sample.json", "removal-pair.json", "effecterase.json", "e2fgvi.json")),
    }
    for path in sorted(path for path in result_json if path.is_file()):
        print(path.relative_to(run_root))
        display(JSON(json.loads(path.read_text())))

print("Repository:", ROOT)


## Canonical one-clip comparison: all four inpainters

The watermark input cannot be prepared, so no remover is selectively presented as if it had run. All four inpainters carry the same explicit status.

| Inpainting method | Status | Reason |
|---|---:|---|
| ProPainter | blocked | official weight server unavailable and required release prompt inputs are missing |
| DiffuEraser | blocked | official weight server unavailable and required release prompt inputs are missing |
| EffectErase | blocked | official weight server unavailable and required release prompt inputs are missing |
| E2FGVI-HQ | blocked | official weight server unavailable and required release prompt inputs are missing |

Watermark method: **LVMark**.


## Watermark evaluation results: all four inpainters

No authentic watermark evaluation can be reported for this method. This is a release-resource blocker, not a missing local or Zeppelin execution.

| Inpainting method | Evaluation status | Reason |
|---|---:|---|
| ProPainter | not run | The released inference script refers to prompt images and a prompt/key JSON that are not included. Its official 2.05 GB weight archive was served to Zeppelin at about 25 KB/s, implying more than 24 hours for the download, so an authentic native watermarked sample could not be generated in this run. |
| DiffuEraser | not run | The released inference script refers to prompt images and a prompt/key JSON that are not included. Its official 2.05 GB weight archive was served to Zeppelin at about 25 KB/s, implying more than 24 hours for the download, so an authentic native watermarked sample could not be generated in this run. |
| EffectErase | not run | The released inference script refers to prompt images and a prompt/key JSON that are not included. Its official 2.05 GB weight archive was served to Zeppelin at about 25 KB/s, implying more than 24 hours for the download, so an authentic native watermarked sample could not be generated in this run. |
| E2FGVI-HQ | not run | The released inference script refers to prompt images and a prompt/key JSON that are not included. Its official 2.05 GB weight archive was served to Zeppelin at about 25 KB/s, implying more than 24 hours for the download, so an authentic native watermarked sample could not be generated in this run. |

**Required to proceed:** The complete official LVMark weight archive plus the missing prompt/key inputs, or a documented equivalent input record. A local run helps only if those resources are supplied.

See the [consolidated 44-case record](../../evidence/watermark-evaluation-44.json) for the complete runnable matrix and blocker provenance.


In [ ]:
LVMARK = Path("/path/to/TIFS26_lvmark")
LVMARK_PYTHON = LVMARK / ".venv/bin/python"
WEIGHTS = Path("/path/to/lvmark_weights")
DYNAMICRAFTER_CKPT = Path("/path/to/dynamicrafter_512_v1/model.ckpt")
PROMPT_DIR = Path("/path/to/authors/prompts/512_ours")
PROMPT_JSON = Path("/path/to/authors/prompts_with_keys120_32bit.json")
DIFFUERASER = Path("/path/to/DiffuEraser")
DIFFUERASER_PYTHON = DIFFUERASER / ".venv/bin/python"
GPU = "0"
RUN_ROOT = ROOT / "runs/remaining-watermarks/lvmark/native-32bit"
NATIVE = RUN_ROOT / "native-generation"
RUN_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
require_commit(LVMARK, "da7ad67ba2d86ae9918dce2b42d3736e335561be")
for path, label in [
    (WEIGHTS / "vae_decoder.pth", "LVMark VAE decoder"),
    (WEIGHTS / "mapping_network.pth", "LVMark mapping network"),
    (WEIGHTS / "decoding_network.pth", "LVMark message decoder"),
    (DYNAMICRAFTER_CKPT, "DynamiCrafter checkpoint"),
    (PROMPT_DIR, "authors' prompt image directory"),
    (PROMPT_JSON, "authors' prompt/key JSON"),
]: require(path, label)


In [ ]:
if not list(NATIVE.rglob("*.avi")):
    env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = GPU
    run([
        LVMARK_PYTHON, LVMARK / "DynamiCrafter/scripts/evaluation/inference_watermark.py",
        "--seed", "42", "--ckpt_path", DYNAMICRAFTER_CKPT,
        "--config", LVMARK / "DynamiCrafter/configs/inference_512_v1.0_wm.yaml",
        "--savedir", NATIVE, "--n_samples", "1", "--bs", "1",
        "--height", "320", "--width", "512", "--unconditional_guidance_scale", "7.5",
        "--ddim_steps", "50", "--ddim_eta", "1.0", "--prompt_dir", PROMPT_DIR,
        "--text_input", "--video_length", "16", "--frame_stride", "24",
        "--timestep_spacing", "uniform_trailing", "--guidance_rescale", "0.7",
        "--int_dimension", "128", "--lr_mult", "1", "--phi_dimension", "32",
        "--vae_decoder_path", WEIGHTS / "vae_decoder.pth",
        "--mapping_path", WEIGHTS / "mapping_network.pth",
        "--msg_decoder_path", WEIGHTS / "decoding_network.pth",
        "--json_file", PROMPT_JSON, "--affine_list", LVMARK / "layer_noise_0.5.txt",
        "--bit_capacity", "32", "--bch_bits", "14",
    ], cwd=LVMARK / "DynamiCrafter", env=env)
WATERMARKED_VIDEO = sorted(NATIVE.rglob("*.avi"))[0]
BCH = sorted(NATIVE.rglob("bch.pkl"))[0]
ALIGNED_MASK = RUN_ROOT / "aligned-mask.mp4"
print(WATERMARKED_VIDEO, BCH)


In [ ]:
run([
    DIFFUERASER_PYTHON, SCRIPTS / "make_native_removal_mask.py", "center-box",
    "--video", WATERMARKED_VIDEO, "--output-video", ALIGNED_MASK,
    "--output-frames", RUN_ROOT / "aligned-mask-frames",
])


## Run all four inpainting methods on the frozen input

The preparation cell freezes hashes, frame count, dimensions, FPS, mask foreground counts, provenance, and any short-video padding. Every remover below consumes that frozen input and mask.


In [ ]:
ATTACK_RUN = RUN_ROOT / "attack"
if not (ATTACK_RUN / "sample.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "prepare_watermark_removal_sample.py", "prepare",
        "--source-video", WATERMARKED_VIDEO, "--source-mask", ALIGNED_MASK,
        "--output", ATTACK_RUN, "--watermark-method", "LVMark",
        "--sample-id", "first-authors-prompt-32bit", "--sample-source", "LVMark authors' prompt/key data and native DynamiCrafter pipeline",
        "--source-kind", "authors_generated",
    ])
else:
    print("Prepared sample already exists:", ATTACK_RUN / "sample.json")


In [ ]:
if not (ATTACK_RUN / "removal-pair.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "run_watermark_removal_pair.py",
        "--run", ATTACK_RUN, "--diffueraser", DIFFUERASER,
        "--diffueraser-python", DIFFUERASER_PYTHON,
        "--device", GPU, "--seed", "2025",
    ])
else:
    print("Both removal outputs already exist:", ATTACK_RUN / "removal-pair.json")


In [ ]:
EFFECTERASE = Path(os.environ.get("EFFECTERASE", "/mnt/data_fast/smayan/research/EffectErase"))
EFFECTERASE_PYTHON = Path(os.environ.get("EFFECTERASE_PYTHON", str(EFFECTERASE / ".venv/bin/python")))
E2FGVI = Path(os.environ.get("E2FGVI", "/mnt/data_fast/smayan/research/E2FGVI"))
E2FGVI_CHECKPOINT = Path(os.environ.get(
    "E2FGVI_CHECKPOINT", str(E2FGVI / "release_model/E2FGVI-HQ-CVPR22.pth")
))
E2FGVI_PYTHON = Path(os.environ.get("E2FGVI_PYTHON", str(DIFFUERASER_PYTHON)))

if not (ATTACK_RUN / "effecterase.json").is_file():
    effect_env = os.environ.copy(); effect_env["CUDA_VISIBLE_DEVICES"] = GPU
    run([
        EFFECTERASE_PYTHON, SCRIPTS / "run_prepared_effecterase.py",
        "--source", EFFECTERASE, "--run", ATTACK_RUN, "--device", GPU,
    ], env=effect_env)
else:
    print("EffectErase output already exists:", ATTACK_RUN / "effecterase.json")

if not (ATTACK_RUN / "e2fgvi.json").is_file():
    run([
        E2FGVI_PYTHON, SCRIPTS / "run_prepared_e2fgvi.py",
        "--source", E2FGVI, "--checkpoint", E2FGVI_CHECKPOINT,
        "--run", ATTACK_RUN, "--python", E2FGVI_PYTHON, "--device", GPU,
    ])
else:
    print("E2FGVI-HQ output already exists:", ATTACK_RUN / "e2fgvi.json")


In [ ]:
run([
    LVMARK_PYTHON, SCRIPTS / "run_lvmark_detection.py",
    "--source", LVMARK, "--decoder", WEIGHTS / "decoding_network.pth", "--bch", BCH,
    "--output", RUN_ROOT / "detection",
    "--case", f"original={ATTACK_RUN / 'input-watermarked.mp4'}",
    "--case", f"propainter={ATTACK_RUN / 'propainter.mp4'}",
    "--case", f"diffueraser={ATTACK_RUN / 'diffueraser.mp4'}",
    "--case", f"effecterase={ATTACK_RUN / 'effecterase.mp4'}",
    "--case", f"e2fgvi={ATTACK_RUN / 'e2fgvi.mp4'}",
])


In [ ]:
show_results(RUN_ROOT)
